# Deney 1 Frame Seçimi — Güncel MediaPipe Tasks API

Bu notebook:

- Google Drive'ı bağlar.
- Daha önce çıkarılmış `real_frames` ve `fake_frames` görüntülerini okur.
- Yüz bulunan görüntüleri MediaPipe **Tasks Face Detector** ile seçer.
- Genel toplam **3000 frame** üretir:
  - **1500 Real**
  - **1500 Fake**
- Her sınıfı kendi içinde:
  - `%80 train`
  - `%10 validation`
  - `%10 test`
  olarak düzenler.
- Kaynak `train/val/test` ayrımını korur.
- Kaynak klasörlerden hiçbir dosya silmez veya taşımaz.
- Yalnızca `OUTPUT_ROOT` içindeki `Real` ve `Fake` çıktı klasörlerini temizleyip yeniden oluşturur.
- Metadata CSV ve yüz bulunamayan görüntüler için log üretir.

> Bu sürüm eski `mp.solutions` arayüzünü kullanmaz. Güncel `mp.tasks.vision.FaceDetector` arayüzünü kullanır.


## 1. Kurulum

Aşağıdaki hücre resmi MediaPipe paketini kurar. `--no-deps` kullanıldığı için Colab'daki NumPy, TensorFlow, OpenCV ve protobuf sürümlerini değiştirmez.

Hücreyi çalıştırdıktan sonra MediaPipe daha önce farklı bir sürümle yüklenmişse:

**Çalışma zamanı → Oturumu yeniden başlat**

seçeneğini bir kez kullan ve notebook'u baştan çalıştır.


In [1]:
# Resmi MediaPipe sürümünü, Colab paketlerini bozmadan kur.
!pip uninstall -y mediapipe
!pip install -q --no-deps mediapipe==0.10.35


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 63.2 MB/s eta 0:00:00


In [2]:
import mediapipe as mp

print("MediaPipe sürümü:", mp.__version__)
print("Tasks var mı:", hasattr(mp, "tasks"))

MediaPipe sürümü: 0.10.35
Tasks var mı: True


## 2. Drive Bağlantısı ve Kütüphaneler


In [3]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [4]:
import os
import csv
import cv2
import json
import shutil
import random
import urllib.request
from pathlib import Path

import numpy as np
import mediapipe as mp
from tqdm.auto import tqdm

print("MediaPipe sürümü:", mp.__version__)
print("OpenCV sürümü:", cv2.__version__)
print("NumPy sürümü:", np.__version__)
print("MediaPipe Tasks mevcut mu:", hasattr(mp, "tasks"))

if not hasattr(mp, "tasks"):
    raise RuntimeError(
        "MediaPipe Tasks API yüklenmemiş görünüyor. "
        "Çalışma zamanını yeniden başlatıp notebook'u baştan çalıştır."
    )


MediaPipe sürümü: 0.10.35
OpenCV sürümü: 4.13.0
NumPy sürümü: 2.0.2
MediaPipe Tasks mevcut mu: True


## 3. Ayarlar ve Klasör Yolları

Aşağıdaki üç yolu kendi Drive klasör yapına göre kontrol et.

Kaynak klasörler sadece okunur:

- `REAL_FRAMES_ROOT`
- `FAKE_FRAMES_ROOT`

Silme işlemi yalnızca `OUTPUT_ROOT/Real` ve `OUTPUT_ROOT/Fake` klasörlerine uygulanır.


In [18]:
# ============================================================
# DRIVE YOLLARI — KENDİ KLASÖR ADINA GÖRE KONTROL ET
# ============================================================

PROJECT_ROOT = "/content/drive/MyDrive/AISC DeepFake Çalışmaları/FaceForensics_Projesi"

REAL_FRAMES_ROOT = os.path.join(PROJECT_ROOT, "real_frames")
FAKE_FRAMES_ROOT = os.path.join(PROJECT_ROOT, "fake_frames")

OUTPUT_ROOT = "/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Deney 1 Frame"

# ============================================================
# HEDEF SAYILAR
# ============================================================

TOTAL_TARGET = 3000
TARGET_PER_CLASS = TOTAL_TARGET // 2  # 1500 Real + 1500 Fake

TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

TARGETS_PER_SPLIT = {
    "train": 1200,
    "val": 150,
    "test": 150,
}

FACE_MIN_CONFIDENCE = 0.50
RANDOM_SEED = 42
IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

METADATA_CSV_PATH = os.path.join(OUTPUT_ROOT, "secim_metadata.csv")
NO_FACE_LOG_PATH = os.path.join(OUTPUT_ROOT, "yuzsuz_veya_okunamayan_frameler.csv")

# MediaPipe Tasks Face Detector model dosyası
MODEL_PATH = "/content/blaze_face_short_range.tflite"
MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "face_detector/blaze_face_short_range/float16/latest/"
    "blaze_face_short_range.tflite"
)

# Sayısal doğrulamalar
assert TOTAL_TARGET == 3000
assert TARGET_PER_CLASS == 1500
assert sum(TARGETS_PER_SPLIT.values()) == TARGET_PER_CLASS
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9

print("Real kaynak:", REAL_FRAMES_ROOT)
print("Fake kaynak:", FAKE_FRAMES_ROOT)
print("Çıktı:", OUTPUT_ROOT)
print("Split hedefleri:", TARGETS_PER_SPLIT)


Real kaynak: /content/drive/MyDrive/AISC DeepFake Çalışmaları/FaceForensics_Projesi/real_frames
Fake kaynak: /content/drive/MyDrive/AISC DeepFake Çalışmaları/FaceForensics_Projesi/fake_frames
Çıktı: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Deney 1 Frame
Split hedefleri: {'train': 1200, 'val': 150, 'test': 150}


## 4. Güvenlik ve Kaynak Klasör Kontrolleri


In [6]:
def normalize_path(path):
    return os.path.realpath(os.path.abspath(path))

real_root_norm = normalize_path(REAL_FRAMES_ROOT)
fake_root_norm = normalize_path(FAKE_FRAMES_ROOT)
output_root_norm = normalize_path(OUTPUT_ROOT)

# Çıktı yolu kaynak klasörlerle aynı veya onların içinde olamaz.
assert output_root_norm != real_root_norm
assert output_root_norm != fake_root_norm
assert not output_root_norm.startswith(real_root_norm + os.sep)
assert not output_root_norm.startswith(fake_root_norm + os.sep)

for root in [REAL_FRAMES_ROOT, FAKE_FRAMES_ROOT]:
    if not os.path.isdir(root):
        raise FileNotFoundError(f"Kaynak klasör bulunamadı: {root}")

    for split in ["train", "val", "test"]:
        split_path = os.path.join(root, split)
        if not os.path.isdir(split_path):
            raise FileNotFoundError(f"Kaynak split klasörü bulunamadı: {split_path}")

print("Kaynak klasörler bulundu.")
print("Güvenlik kontrolleri başarılı.")
print("Kaynak klasörlerden hiçbir veri silinmeyecek.")


Kaynak klasörler bulundu.
Güvenlik kontrolleri başarılı.
Kaynak klasörlerden hiçbir veri silinmeyecek.


## 5. MediaPipe Face Detector Modelini İndirme


In [7]:
if not os.path.isfile(MODEL_PATH):
    print("Face Detector modeli indiriliyor...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)

if not os.path.isfile(MODEL_PATH) or os.path.getsize(MODEL_PATH) == 0:
    raise RuntimeError("Face Detector model dosyası indirilemedi.")

print("Model hazır:", MODEL_PATH)
print("Model boyutu:", os.path.getsize(MODEL_PATH), "bayt")


Face Detector modeli indiriliyor...
Model hazır: /content/blaze_face_short_range.tflite
Model boyutu: 229746 bayt


## 6. Çıktı Klasörlerini Hazırlama


In [8]:
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# DİKKAT:
# Yalnızca OUTPUT_ROOT içindeki Real ve Fake klasörleri temizlenir.
# real_frames ve fake_frames kaynak klasörlerine dokunulmaz.
for class_name in ["Real", "Fake"]:
    class_output = os.path.join(OUTPUT_ROOT, class_name)

    if os.path.exists(class_output):
        shutil.rmtree(class_output)

    for split in ["train", "val", "test"]:
        os.makedirs(
            os.path.join(class_output, split),
            exist_ok=True
        )

print("Çıktı klasörleri temizlendi ve yeniden oluşturuldu.")


Çıktı klasörleri temizlendi ve yeniden oluşturuldu.


## 7. Yardımcı Fonksiyonlar


In [12]:
def find_all_images(root_dir):
    """Bir klasörün altındaki desteklenen bütün görüntüleri döndürür."""
    paths = []

    for dirpath, _, filenames in os.walk(root_dir):
        for filename in filenames:
            if filename.lower().endswith(IMG_EXTENSIONS):
                paths.append(os.path.join(dirpath, filename))

    return sorted(paths)


def has_face(image_path, detector):
    """Görüntü okunabiliyor ve en az bir yüz içeriyorsa True döndürür."""
    image_bgr = cv2.imread(image_path)

    if image_bgr is None:
        return False, "goruntu_okunamadi"

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    image_rgb = np.ascontiguousarray(image_rgb)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=image_rgb
    )

    detection_result = detector.detect(mp_image)

    if detection_result.detections:
        return True, "yuz_bulundu"

    return False, "yuz_bulunamadi"


def select_face_frames(
    frame_pool,
    target_count,
    detector,
    rejected_rows,
    class_name,
    split_name,
    seed
):
    candidates = frame_pool.copy()
    random.Random(seed).shuffle(candidates)

    selected = []

    progress = tqdm(
        candidates,
        desc=f"{class_name}/{split_name} yüz kontrolü"
    )

    for image_path in progress:

        if len(selected) >= target_count:
            break

        face_found, reason = has_face(image_path, detector)

        if face_found:
            selected.append(image_path)
        else:
            rejected_rows.append({
                "sinif": class_name,
                "split": split_name,
                "dosya": image_path,
                "neden": reason,
            })

        progress.set_postfix(
            secilen=len(selected),
            hedef=target_count,
            bu_split_reddedilen=(
                progress.n - len(selected)
            )
        )

    progress.close()

    if len(selected) < target_count:
        raise RuntimeError(
            f"{class_name}/{split_name} için yeterli yüzlü frame bulunamadı. "
            f"Hedef={target_count}, bulunan={len(selected)}, "
            f"toplam_havuz={len(frame_pool)}"
        )

    return selected

def create_unique_filename(class_name, split_name, index, src_path):
    extension = os.path.splitext(src_path)[1].lower()

    if extension not in IMG_EXTENSIONS:
        extension = ".jpg"

    return f"{class_name.lower()}_{split_name}_{index:05d}{extension}"


def copy_selected_frames(
    selected_paths,
    class_name,
    split_name,
    metadata_rows
):
    """Seçilen görüntüleri çıktı klasörüne kopyalar."""
    destination_dir = os.path.join(
        OUTPUT_ROOT,
        class_name,
        split_name
    )

    for index, src_path in enumerate(selected_paths):
        filename = create_unique_filename(
            class_name,
            split_name,
            index,
            src_path
        )

        dst_path = os.path.join(destination_dir, filename)

        # copy2 kaynak görüntüyü silmez; yalnızca kopyalar.
        shutil.copy2(src_path, dst_path)

        metadata_rows.append({
            "sinif": class_name,
            "split": split_name,
            "orijinal_yol": src_path,
            "yeni_yol": dst_path,
            "dosya_adi": filename,
        })


def count_images(folder):
    return sum(
        1
        for filename in os.listdir(folder)
        if filename.lower().endswith(IMG_EXTENSIONS)
    )


## 8. Kaynak Görüntü Havuzlarını Toplama


In [10]:
source_pools = {
    "Real": {
        split: find_all_images(
            os.path.join(REAL_FRAMES_ROOT, split)
        )
        for split in ["train", "val", "test"]
    },
    "Fake": {
        split: find_all_images(
            os.path.join(FAKE_FRAMES_ROOT, split)
        )
        for split in ["train", "val", "test"]
    },
}

for class_name in ["Real", "Fake"]:
    print(f"\n{class_name} kaynak sayıları")

    for split_name in ["train", "val", "test"]:
        pool_count = len(source_pools[class_name][split_name])
        target_count = TARGETS_PER_SPLIT[split_name]

        print(
            f"  {split_name}: {pool_count} görüntü "
            f"| hedef: {target_count}"
        )

        if pool_count < target_count:
            raise RuntimeError(
                f"{class_name}/{split_name} klasöründe "
                f"hedef sayıdan daha az görüntü var."
            )



Real kaynak sayıları
  train: 21705 görüntü | hedef: 1200
  val: 2745 görüntü | hedef: 150
  test: 2742 görüntü | hedef: 150

Fake kaynak sayıları
  train: 135550 görüntü | hedef: 1200
  val: 16903 görüntü | hedef: 150
  test: 16932 görüntü | hedef: 150


## 9. Güncel MediaPipe Tasks API ile Yüzlü Frame Seçimi


In [13]:
rejected_rows = []
selected_frames = {"Real": {}, "Fake": {}}

BaseOptions = mp.tasks.BaseOptions
FaceDetector = mp.tasks.vision.FaceDetector
FaceDetectorOptions = mp.tasks.vision.FaceDetectorOptions
RunningMode = mp.tasks.vision.RunningMode

detector_options = FaceDetectorOptions(
    base_options=BaseOptions(
        model_asset_path=MODEL_PATH
    ),
    running_mode=RunningMode.IMAGE,
    min_detection_confidence=FACE_MIN_CONFIDENCE
)

with FaceDetector.create_from_options(detector_options) as detector:

    for class_index, class_name in enumerate(["Real", "Fake"]):
        for split_index, split_name in enumerate(
            ["train", "val", "test"]
        ):
            seed = (
                RANDOM_SEED
                + class_index * 100
                + split_index
            )

            selected_frames[class_name][split_name] = (
                select_face_frames(
                    frame_pool=source_pools[class_name][split_name],
                    target_count=TARGETS_PER_SPLIT[split_name],
                    detector=detector,
                    rejected_rows=rejected_rows,
                    class_name=class_name,
                    split_name=split_name,
                    seed=seed,
                )
            )

for class_name in ["Real", "Fake"]:
    print(f"\n{class_name} seçilen sayıları")

    for split_name in ["train", "val", "test"]:
        print(
            f"  {split_name}: "
            f"{len(selected_frames[class_name][split_name])}"
        )

print(
    "\nYüz bulunamadığı veya okunamadığı için atlanan:",
    len(rejected_rows)
)


Real/train yüz kontrolü:   0%|          | 0/21705 [00:00<?, ?it/s]

Real/val yüz kontrolü:   0%|          | 0/2745 [00:00<?, ?it/s]

Real/test yüz kontrolü:   0%|          | 0/2742 [00:00<?, ?it/s]

Fake/train yüz kontrolü:   0%|          | 0/135550 [00:00<?, ?it/s]

Fake/val yüz kontrolü:   0%|          | 0/16903 [00:00<?, ?it/s]

Fake/test yüz kontrolü:   0%|          | 0/16932 [00:00<?, ?it/s]


Real seçilen sayıları
  train: 1200
  val: 150
  test: 150

Fake seçilen sayıları
  train: 1200
  val: 150
  test: 150

Yüz bulunamadığı veya okunamadığı için atlanan: 206


## 10. Seçilen Görüntüleri Kopyalama


In [14]:
metadata_rows = []

for class_name in ["Real", "Fake"]:
    for split_name in ["train", "val", "test"]:
        copy_selected_frames(
            selected_paths=selected_frames[class_name][split_name],
            class_name=class_name,
            split_name=split_name,
            metadata_rows=metadata_rows,
        )

print("Toplam kopyalanan frame:", len(metadata_rows))


Toplam kopyalanan frame: 3000


## 11. Metadata ve Log Dosyalarını Kaydetme


In [15]:
with open(
    METADATA_CSV_PATH,
    "w",
    newline="",
    encoding="utf-8-sig"
) as csv_file:
    writer = csv.DictWriter(
        csv_file,
        fieldnames=[
            "sinif",
            "split",
            "orijinal_yol",
            "yeni_yol",
            "dosya_adi",
        ]
    )
    writer.writeheader()
    writer.writerows(metadata_rows)

with open(
    NO_FACE_LOG_PATH,
    "w",
    newline="",
    encoding="utf-8-sig"
) as csv_file:
    writer = csv.DictWriter(
        csv_file,
        fieldnames=[
            "sinif",
            "split",
            "dosya",
            "neden",
        ]
    )
    writer.writeheader()
    writer.writerows(rejected_rows)

print("Metadata:", METADATA_CSV_PATH)
print("Reddedilen frame logu:", NO_FACE_LOG_PATH)


Metadata: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1 Frame/secim_metadata.csv
Reddedilen frame logu: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1 Frame/yuzsuz_veya_okunamayan_frameler.csv


## 12. Kesin Sonuç Doğrulaması


In [16]:
actual_total = 0

print("=" * 70)
print("ÇIKTI DOĞRULAMA")
print("=" * 70)

for class_name in ["Real", "Fake"]:
    class_total = 0

    for split_name in ["train", "val", "test"]:
        folder = os.path.join(
            OUTPUT_ROOT,
            class_name,
            split_name
        )

        actual = count_images(folder)
        expected = TARGETS_PER_SPLIT[split_name]

        print(
            f"{class_name:4s}/{split_name:5s}: "
            f"{actual:4d} | beklenen: {expected:4d}"
        )

        assert actual == expected, (
            f"{class_name}/{split_name} sayısı yanlış. "
            f"Bulunan={actual}, beklenen={expected}"
        )

        class_total += actual
        actual_total += actual

    assert class_total == TARGET_PER_CLASS
    print(f"{class_name} toplam: {class_total}\n")

assert actual_total == TOTAL_TARGET
assert len(metadata_rows) == TOTAL_TARGET

source_paths = [
    row["orijinal_yol"]
    for row in metadata_rows
]

assert len(source_paths) == len(set(source_paths)), (
    "Aynı kaynak görüntü birden fazla kez seçilmiş."
)

print(f"GENEL TOPLAM: {actual_total}")
print("Dağılım: 1500 Real + 1500 Fake")
print("Her sınıf: 1200 train + 150 val + 150 test")
print("Kaynak görüntüler silinmedi; yalnızca kopyalandı.")


ÇIKTI DOĞRULAMA
Real/train: 1200 | beklenen: 1200
Real/val  :  150 | beklenen:  150
Real/test :  150 | beklenen:  150
Real toplam: 1500

Fake/train: 1200 | beklenen: 1200
Fake/val  :  150 | beklenen:  150
Fake/test :  150 | beklenen:  150
Fake toplam: 1500

GENEL TOPLAM: 3000
Dağılım: 1500 Real + 1500 Fake
Her sınıf: 1200 train + 150 val + 150 test
Kaynak görüntüler silinmedi; yalnızca kopyalandı.


## 13. İşlem Özeti


In [17]:
summary = {
    "genel_toplam": len(metadata_rows),
    "real": {
        split: len(selected_frames["Real"][split])
        for split in ["train", "val", "test"]
    },
    "fake": {
        split: len(selected_frames["Fake"][split])
        for split in ["train", "val", "test"]
    },
    "reddedilen_frame_sayisi": len(rejected_rows),
    "output_root": OUTPUT_ROOT,
}

print(json.dumps(summary, ensure_ascii=False, indent=2))


{
  "genel_toplam": 3000,
  "real": {
    "train": 1200,
    "val": 150,
    "test": 150
  },
  "fake": {
    "train": 1200,
    "val": 150,
    "test": 150
  },
  "reddedilen_frame_sayisi": 206,
  "output_root": "/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1 Frame"
}


In [19]:
import pandas as pd

df = pd.read_csv(METADATA_CSV_PATH)

fake = df[df["sinif"]=="Fake"]

print(fake.groupby("split").size())

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Deney 1 Frame/secim_metadata.csv'